we tried replicating the bs and buspas like how its mentioned in the slides shared by bart on canvas. for bus it was ok but for bs the thesis said they changed it up a bit to make it comparable but we just copied how it is from the slides. so we dk specifically what changed but thats an assumption we are going with whatever we made below.

also we first tried with actual time series stocks, sales, stores dataset and they were random and thus unable to make fair comparison so we tried one of the ideas from the thesis about trying with synthetic data and forcing out subgroups with varying number of descriptors


In [1]:
import time, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

N, T, LITS, OUT, SIG2, DMU = 500, 24, 3, 0.2, 5, 1
D_MAX, Q, W, W_WIDE, U, MIN_COV = 3, 5, 20, 200, 150, 0.01
K_VALUES = [8, 16, 24, 32]
REPS = 10

## Synthetic data

In [2]:
def cauchy_walk(rng, x0, size):
    steps = x0 + 6 * rng.standard_cauchy(size)
    return np.cumsum(steps, axis=-1)

def make_data(k, seed):
    rng = np.random.default_rng(seed)
    A = rng.integers(0, 2, size=(N, k))
    Y = cauchy_walk(rng, -0.5, (N, T))
    true_mask = np.all(A[:, :LITS] == 0, axis=1)
    idx = np.where(true_mask)[0]
    members = idx[rng.random(len(idx)) >= OUT]
    base = cauchy_walk(rng, -0.5 + DMU, T)
    noise = np.cumsum(rng.normal(0, np.sqrt(SIG2), (len(members), T)), axis=1)
    Y[members] = base + noise
    return A, Y, true_mask

## Quality measure

In [9]:
class Quality:
    def __init__(self, Y):
        self.D = cdist(Y, Y)
        self.rowsum = self.D.sum(1)
        self.cache = {}

    def __call__(self, m):
        key = m.tobytes()
        if key not in self.cache:
            n = m.sum()
            if n < max(2, MIN_COV * len(m)) or n == len(m):
                self.cache[key] = None
            else:
                s = self.D @ m
                within = s[m].sum()
                beta = within / (n * (n - 1))
                alpha = (self.rowsum[m].sum() - within) / (n * (len(m) - n))
                self.cache[key] = alpha / (beta + 1)
        return self.cache[key]

def mask_of(A, desc):
    m = np.ones(len(A), bool)
    for j, v in desc:
        m &= A[:, j] == v
    return m

## Beam search

In [4]:
def beam_search(A, qual, w=W):
    k = A.shape[1]
    conds = [(j, v) for j in range(k) for v in (0, 1)]
    results, seeds, seen = {}, [()], set()
    for _ in range(D_MAX):
        beam = []
        for seed in seeds:
            used = {j for j, _ in seed}
            for c in conds:
                if c[0] in used:
                    continue
                desc = tuple(sorted(seed + (c,)))
                if desc in seen:
                    continue
                seen.add(desc)
                m = mask_of(A, desc)
                phi = qual(m)
                if phi is not None:
                    results[desc] = (phi, m)
                    beam.append((phi, desc))
        beam.sort(reverse=True)
        seeds = [d for _, d in beam[:w]]
    top = sorted(results.values(), key=lambda x: -x[0])[:Q]
    return top, len(seen)

## Bottom-up search

In [13]:
def bottom_up_search(A, qual):
    D = qual.D
    iu = np.triu_indices(len(D), 1)
    print(f" iu = {iu}")
    flat = D[iu]
    print(f"flat = {flat}")
    best = np.argpartition(flat, U)[:U]
    pairs = zip(iu[0][best], iu[1][best])
    cands = set()
    for i, j in pairs:
        shared = [(a, A[i, a]) for a in range(A.shape[1]) if A[i, a] == A[j, a]]
        for d in range(1, D_MAX + 1):
            cands.update(itertools.combinations(shared, d))
    results = []
    for desc in cands:
        m = mask_of(A, desc)
        phi = qual(m)
        if phi is not None:
            results.append((phi, m))
    top = sorted(results, key=lambda x: -x[0])[:Q]
    return top, len(cands)

## Hit check

In [6]:
def found(top, true_mask, thr=0.8):
    for _, m in top:
        inter = (m & true_mask).sum()
        union = (m | true_mask).sum()
        if inter / union >= thr:
            return True
    return False

## Experiment

In [14]:
rows = []
for k in K_VALUES:
    for r in range(REPS):
        A, Y, true_mask = make_data(k, seed=1000 * k + r)
        q_bs = Quality(Y)
        t = time.time(); top_bs, n_bs = beam_search(A, q_bs); t_bs = time.time() - t
        q_bus = Quality(Y)
        t = time.time(); top_bus, n_bus = bottom_up_search(A, q_bus); t_bus = time.time() - t
        t = time.time(); top_wide, n_wide = beam_search(A, Quality(Y), W_WIDE); t_wide = time.time() - t
        hb, hu = found(top_bs, true_mask), found(top_bus, true_mask)
        rows.append(dict(k=k, rep=r, bs=hb, bus=hu, union=hb or hu,
                         bs_wide=found(top_wide, true_mask),
                         t_bs=t_bs, t_bus=t_bus, t_wide=t_wide,
                         n_bs=n_bs, n_bus=n_bus, n_wide=n_wide,
                         best_bs=top_bs[0][0], best_bus=top_bus[0][0]))
        break
    print(k, "done")
    break

res = pd.DataFrame(rows)

 iu = (array([  0,   0,   0, ..., 497, 497, 498]), array([  1,   2,   3, ..., 498, 499, 499]))
flat = [1086.38162949  360.84555111  486.21624534 ...  243.98339179  142.09033121
  247.44943561]
8 done


## Results

In [8]:
summary = res.groupby("k").agg(
    bs=("bs", "mean"), bus=("bus", "mean"), union=("union", "mean"), bs_wide=("bs_wide", "mean"),
    only_bs=("bs", lambda s: (s & ~res.loc[s.index, "bus"]).sum()),
    only_bus=("bus", lambda s: (s & ~res.loc[s.index, "bs"]).sum()),
    t_bs=("t_bs", "mean"), t_bus=("t_bus", "mean"), t_wide=("t_wide", "mean"),
    n_bs=("n_bs", "mean"), n_bus=("n_bus", "mean"), n_wide=("n_wide", "mean"),
)
summary

,bs,bus,union,bs_wide,only_bs,only_bus,t_bs,t_bus,t_wide,n_bs,n_bus,n_wide
k,,,,,,,,,,,,
8,0.9,0.9,0.9,0.9,0,0,0.016473,0.019933,0.037388,298.1,294.2,576.0
16,0.5,0.7,0.7,0.7,0,2,0.050298,0.178018,0.198559,949.1,3242.1,3597.6
24,0.3,0.5,0.5,0.4,0,2,0.080524,0.665806,0.387706,1596.3,11781.1,7386.3
32,0.2,0.7,0.7,0.3,0,5,0.117619,1.512732,0.621887,2269.8,29236.4,11447.3


about the results...

1. At k = 8, all methods find the true subgroup 9 times out of 10.
2. As k grows, BS drops sharply: 0.5 at k = 16, 0.3 at k = 24, 0.2 at k = 32.
3. BUS holds up better: 0.7, 0.5 and 0.7.
4. But BUS's cost explodes. At k = 32 it checks about 29k candidate descriptions, while BS checks about 2.3k That's 10x or more
5. At k = 16, the wide-beam BS matches BUS (0.7) with a similar budget. So there, the BUS advantage seems to come from checking more candidates. At k = 32, wide BS reaches only 0.3, with about 11k checks.
6. BS never found a subgroup that BUS missed (0 cases out of 40). So a simple ensemble learning of doing a combination of both searches adds nothing here(and ensemble approac is what bart's thesis suggested)
7. With 50% outliers (one of the thesis settings), the true subgroup often wasn't the highest-scoring subgroup. So neither method could actually find it. We lowered outliers to 20% for this reason(but we dont really understand it much, we can play round w it)

main qs:

-the thesis claims that at more descriptors bus fails and we tried it and it makes sense. they suggested an ensemble strategy but a simpler one won't help. the thesis also suggest smth of a hybrid BUS + BS where we may use BUS mini-clusters as starting points, then explore their descriptions with a beam instead of trying every combination. The goal is BUS-level accuracy at much lower cost. we want to do this as next steps but wondering if we simply redo these experiments mentioned in future work and we end up only validating and finding nothing else does it help for the objective of this course?

-we also have two smaller directions. one is numeric descriptors, since bus needs exact attribute matches so it should struggle there, and that might be the first place where bs or an ensemble actually adds something (in our binary test bs never found anything bus missed). the other is improving the mini-clusters themselves, like also using the 2nd/3rd closest neighbours, or dropping mini-clusters that are already below the dfd significance threshold. would either of these be enough as our own contribution, or should we treat them as small add-ons to the hybrid idea?
   